# Build and check the frozen film sample
Replay all 505 parsed records and recheck 20 original HTML pages. Rebuild the ranked selection and compare it with the frozen film/page pairs. No collection or shared-data writes occur by default.

| Sections | Purpose |
| --- | --- |
| Find links; read and screen HTML | Acquisition, M1 lifecycle, M2.3 noise |
| Choose a page; inspect repeat measurements | M2.6 duplicates |
| Review logs and passport | M1 profiling and source evidence |
| Check arithmetic and frozen table | M2 quality; inputs for M3-M6 in Analysis |


In [1]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
import time
import hashlib
import math
from decimal import Decimal
from collections import Counter
from IPython.display import display

# Keep evidence files needed by parsing, logs and freeze checks.
from tempfile import TemporaryDirectory
from zipfile import ZipFile
project_dir = Path.cwd()
temp = TemporaryDirectory(prefix="ads206_")
with ZipFile(project_dir / "data_compact.zip") as data:
    for name in data.namelist():
        raw_input = name.startswith(("01_raw/candidate_links/", "01_raw/film_measurements/", "01_raw/tmdb_candidates/"))
        frozen_input = name in ["data_passport.json", "02_audit/tmdb_candidates.json", "02_audit/frozen_films.csv",
                               "02_audit/frozen_films_version.json", "02_audit/film_measurements.json"]
        if raw_input or frozen_input:
            data.extract(name, temp.name)
folder = Path(temp.name)
live = False
base = folder / "01_raw/film_measurements"
run_id = (base / "latest_run.txt").read_text(encoding="utf-8").strip()
run = base / run_id
cache_index = json.loads((run / "cache_index.json").read_text(encoding="utf-8"))
duration_low, duration_high, target_per_decade = 0.90, 1.10, 20
headers = {"User-Agent": "ADS206-film-study/0.2 (educational, limited requests)"}
proxy_url = None

film_version = json.loads((folder / "02_audit/frozen_films_version.json").read_text(encoding="utf-8"))
for field, hash_field in [("table", "table_sha256"), ("source", "source_sha256")]:
    assert hashlib.sha256((folder / film_version[field]).read_bytes()).hexdigest() == film_version[hash_field]
frozen_main = pd.read_csv(folder / film_version["table"], encoding="utf-8-sig")
frozen_source = json.loads((folder / film_version["source"]).read_text(encoding="utf-8"))
frozen_selected = [row for row in frozen_source["records"] if row.get("selected_for_main_analysis") is True]


## Read measurement HTML
Read successful cache files first. Live requests retain User-Agent, timeout, throttling, status handling, response hashes, structured logs and exponential retry.

In [2]:
from email.utils import parsedate_to_datetime

def retry_delay(reply, attempt):
    delay = 2 ** attempt
    value = reply.headers.get("Retry-After", "").strip() if reply is not None else ""
    if value.isdecimal():
        delay = max(delay, int(value))
    elif value:
        try:
            until = parsedate_to_datetime(value)
            delay = max(delay, (until - datetime.now(timezone.utc)).total_seconds())
        except (TypeError, ValueError, OverflowError):
            pass  # Malformed header: use the normal backoff.
    if delay > 60:
        raise RuntimeError("Server asked for a longer wait. Stop this run and retry later.")
    return delay

def request(method, url, target, expect_json=False, **kwargs):
    target.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        start = time.perf_counter()
        log = {"time_utc": datetime.now(timezone.utc).isoformat(), "url": url, "method": method, "retry_count": attempt}
        reply = None
        try:
            reply = requests.request(method, url, headers=headers, timeout=(5, 15),
                                     proxies={"https": proxy_url} if proxy_url else None, **kwargs)
            name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ.response")
            (target / name).write_bytes(reply.content)
            log.update(status=reply.status_code, file=name, sha256=hashlib.sha256(reply.content).hexdigest())
            if expect_json and reply.ok:
                try:
                    parsed = reply.json()
                except ValueError:
                    log["error"] = "invalid directory JSON"
        except requests.RequestException as error:
            log["error"] = type(error).__name__
        log["seconds"] = time.perf_counter() - start
        with (target / "requests.jsonl").open("a", encoding="utf-8") as out:
            out.write(json.dumps(log) + "\n")
        time.sleep(1.2)
        if reply is not None and reply.ok:
            if "error" in log:
                raise ValueError(f"Invalid directory JSON from {url}. See {target / 'requests.jsonl'}")
            return parsed if expect_json else reply.content
        if reply is not None and reply.status_code < 500 and reply.status_code not in (408, 429):
            reply.raise_for_status()
        if attempt < 2:
            time.sleep(retry_delay(reply, attempt))
    raise ConnectionError(f"Request failed for {url}. See {target / 'requests.jsonl'}")

def cached_page_path(page_id):
    name = page_id + ".html"
    path = run / name
    return path if path.exists() else run / cache_index.get(name, name)

def fetch(page):
    path = cached_page_path(page["page_id"])
    if path.exists():
        return path.read_bytes()
    if live:
        return request("GET", page["url"], project_dir / "01_raw/film_measurements/live")
    return None


## Match ranked films to directory pages
Retain twenty passing films per decade in vote-ranked order. Within each film, choose the closest runtime ratio and then page ID using full precision.

In [3]:
pool = json.loads((folder / "02_audit/tmdb_candidates.json").read_text(encoding="utf-8"))
links_base = folder / "01_raw/candidate_links"
links_id = (links_base / "latest_run.txt").read_text(encoding="utf-8").strip()
links = json.loads((links_base / links_id / "links.json").read_text(encoding="utf-8"))
assert links["candidate_run"] == pool["run_id"], "Candidate and directory runs differ."
if live:
    # Extend directory discovery only for new candidate pages.
    import html
    for line in ((project_dir / ".env").read_text(encoding="utf-8-sig").splitlines() if (project_dir / ".env").exists() else []):
        if line.startswith("HTTPS_PROXY="):
            proxy_url = line.split("=", 1)[1].strip()
    link_run = links_base / links_id
    logs = [json.loads(line) for line in (link_run / "requests.jsonl").read_text(encoding="utf-8").splitlines()]
    template = next(row["request_body"] for row in reversed(logs)
                    if row.get("status") == 200 and "let $matching" in row.get("request_body", ""))
    endpoint = next(row["url"] for row in reversed(logs) if row.get("method") == "POST")
    groups = {(int(group["decade"]), group.get("candidate_page", 1)) for group in links["decades"]}
    for decade, page_number in sorted({(int(film["decade"][:4]), film["page"]) for film in pool["records"]} - groups):
        titles = [film["title"].strip().lower() for film in pool["records"]
                  if film["decade"] == f"{decade}s" and film["page"] == page_number]
        values = ", ".join('"' + html.escape(title.replace('"', '""'), quote=False) + '"' for title in titles)
        query = re.sub(r"ge \d{4} and xs:integer\(\$year\) lt \d{4}",
                       f"ge {decade} and xs:integer($year) lt {decade + 10}", template)
        query = re.sub(r" = \(.*?\) return \$film", lambda match: " = (" + values + ") return $film", query, count=1)
        rows, offset, total = [], 1, None
        while total is None or offset <= total:
            body = re.sub(r"subsequence\(\$matching, \d+, 50\)", f"subsequence($matching, {offset}, 50)", query)
            name = f"directory_{decade}_candidates_page{page_number}_part{(offset - 1) // 50 + 1}.json"
            cached = link_run / name
            data = json.loads(cached.read_text(encoding="utf-8")) if cached.exists() else None
            if data is None:
                data = request("POST", endpoint, project_dir / "01_raw/candidate_links/live",
                               expect_json=True, data=body.encode("utf-8"))
            if data is None:
                raise ConnectionError(f"Directory request failed: {decade}s, candidate page {page_number}, offset {offset}. See requests.jsonl.")
            group = data["result"]["ochre"]["rows"]
            total = int(group["totalCount"])
            part = group.get("film", [])
            if isinstance(part, dict): part = [part]
            assert isinstance(part, list) and (part or offset > total), "Check directory paging."
            rows.extend(part)
            offset += 50
        assert len(rows) == total, "Directory page count differs."
        links["decades"].append({"decade": decade, "candidate_page": page_number,
            "reported_measurements": total, "returned_rows": len(rows),
            "directory_pages_read": (total + 49) // 50, "rows": rows})
        print(f"Directory {decade}s candidate page {page_number}: {len(rows)} links", flush=True)
pages, seen, link_review = [], set(), []
for group in links["decades"]:
    for page in group["rows"]:
        matches = [film for film in pool["records"] if film["title"].strip().casefold() == str(page["title"]).strip().casefold()
                   and int(film["release_date"][:4]) == int(page["year"])]
        if len(matches) != 1:
            link_review.append({**page, "reason": "No unique candidate with the same title and year"})
        if len(matches) == 1 and page["uuid"] not in seen:
            pages.append({"page_id": page["uuid"], "tmdb_id": matches[0]["tmdb_id"],
                          "url": "https://cinemetrics.uchicago.edu/movie/" + page["uuid"],
                          "directory_director": page.get("director") if isinstance(page.get("director"), str) else ""})
            seen.add(page["uuid"])
order = {film["tmdb_id"]: (film["decade"], film["page"], film["rank_on_page"]) for film in pool["records"]}
pages.sort(key=lambda page: order[page["tmdb_id"]])
inputs = {"candidate_run": pool["run_id"], "links_run": links_id, "films": pool["records"], "pages": pages, "link_review": link_review,
          "rules": {key: pool[key] for key in ["date_start", "date_end", "rating_cutoff", "vote_count_cutoff"]}}
if not live:
    saved_inputs = json.loads((run / "inputs.json").read_text(encoding="utf-8"))
    assert inputs["pages"] == saved_inputs["pages"], "Directory replay changed the recorded page inputs."
films = {film["tmdb_id"]: film for film in inputs["films"]}
print("Mode:", "live requests" if live else "saved responses")
print("Candidate films:", len(films), "Candidate measurement links:", len(inputs["pages"]))

Mode: saved responses
Candidate films: 260 Candidate measurement links: 651


## Extract and screen measurements
Overall buttons must contain one NoS/LEN/ASL/MSL value each. Check shared IMDb/year or exact title/year/director, the inclusive 0.90-1.10 duration band, positive finite fields, partial wording and source accuracy warnings; failures keep their reasons.

In [4]:
def overall_stats(soup):
    stats = {}
    for button in soup.select("button"):
        spans = button.find_all("span", recursive=False)
        if len(spans) == 2:
            label = spans[0].get_text(strip=True).rstrip(":")
            if label in {"NoS", "LEN", "ASL", "MSL"}:
                assert label not in stats, "Repeated overall statistic"
                stats[label] = spans[1].get_text(strip=True)
    assert len(stats) == 4, "Missing overall field"
    return stats

def seconds(value):
    parts = value.split(":")
    return sum(float(part) * 60 ** index for index, part in enumerate(reversed(parts)))

def parse_page(page, raw):
    soup = BeautifulSoup(raw, "html.parser")
    header = soup.find("header")
    if header is None:
        raise ValueError("Missing film header")
    heading = header.get_text(" ", strip=True)
    title = re.search(r"^(.*?)\s*\(\s*(\d{4})", heading)
    assert title, "Check the page heading."
    stats = overall_stats(soup)
    ids = {re.search(r"tt\d+", a["href"]).group()
           for a in soup.find_all("a", href=re.compile(r"imdb\.com/title/tt\d+"))}
    comment = re.search(r"comment:\s*(.*?)\s+NoS\s*:", soup.get_text(" ", strip=True), re.I)
    film = films[page["tmdb_id"]]
    year = int(title[2])
    year_ok = year == int(film["release_date"][:4])
    names = [name.strip().casefold() for name in re.split(r",|;|/|&|\band\b", page["directory_director"])]
    director_ok = any(name.casefold() in names for name in film.get("directors", []))
    title_ok = title[1].strip().casefold() == film["title"].strip().casefold()
    # Match film identity before checking measurement coverage.
    identity = "IMDb + year" if film.get("imdb_id") in ids and year_ok else (
        "title + year + director" if title_ok and year_ok and director_ok else "review")
    record = {**film, **page, "title_html": title[1].strip(), "year_html": year,
        "imdb_ids_html": sorted(ids), "shot_count": int(stats["NoS"].replace(",", "")),
        "measured_duration": stats["LEN"], "asl_seconds": float(stats["ASL"]),
        "msl_seconds": float(stats["MSL"]), "comment": comment[1] if comment else "",
        "identity_check": identity,
        "measurement_check": "review boundaries", "selected_for_main_analysis": False}
    record["measured_seconds"] = seconds(record["measured_duration"])
    # Compare measured seconds with the TMDB reference runtime.
    record["duration_ratio"] = record["measured_seconds"] / (film["runtime_minutes"] * 60) if film.get("runtime_minutes", 0) > 0 else None
    wording = record["title_html"] + " " + record["comment"]
    partial = re.search(r"\b(first|last)\s+(\d+|hour|half)\b|\bsecond half\b|\bexcerpt\b|\bclip\b|\btrailer\b|scene only|volleyball scene", wording, re.I)
    record["measurement_check"] = "partial wording" if partial else "review boundaries"
    record["accuracy_warning"] = bool(re.search(r"not (very )?accurate|inaccurate|rough estimate", record["comment"], re.I))
    record["edition_check"] = "source edition not independently verified"
    record["numeric_check"] = all(math.isfinite(record[key]) and record[key] > 0
        for key in ["shot_count", "asl_seconds", "msl_seconds", "measured_seconds"])
    record.pop("cinemetrics_check", None)
    # Keep all applicable reasons, including invalid coverage and source wording.
    checks = [(record["identity_check"] == "review", "identity review"),
              (record["duration_ratio"] is None or not 0.90 <= record["duration_ratio"] <= 1.10, "outside duration band"),
              (bool(partial), "partial wording"), (not record["numeric_check"], "numeric review"),
              (record["accuracy_warning"], "source accuracy warning")]
    reasons = [reason for failed, reason in checks if failed]
    record["review_reasons"] = reasons
    record["passes_initial_checks"] = not reasons
    return record, stats

# Match each retained page to its original successful request.
compact = json.loads((run / "compact_audit.json").read_text(encoding="utf-8"))
logs = [json.loads(line) for line in (run / "requests.jsonl").read_text(encoding="utf-8").splitlines()]
good_logs = {r["url"]: r for r in logs if r.get("status") == 200}
by_page = {r["page_id"]: r for r in frozen_source["records"]}
sample_ids = {r["page_id"] for r in compact["sample"]}
assert len(sample_ids) == 20 and len(by_page) == len(good_logs) == 505
assert set(good_logs) == {r["url"] for r in by_page.values()}
sample_stats, noise_rows = {}, []
records, chosen = [], {}
for page in inputs["pages"]:
    pid = page["page_id"]
    if pid not in by_page:
        continue
    saved = by_page[pid]
    assert (page["tmdb_id"], page["url"]) == (saved["tmdb_id"], saved["url"])
    # Reparse archived examples. Live mode can fetch the missing bodies.
    if pid in sample_ids or live:
        raw = fetch(page)
        log = good_logs[page["url"]]
        assert page["url"].endswith("/" + pid) and cached_page_path(pid).name == log["file"]
        assert hashlib.sha256(raw).hexdigest() == log["sha256"], f"Response differs from the capture: {pid}"
        parsed, stats = parse_page(page, raw)
        for field in ["shot_count", "measured_duration", "asl_seconds", "msl_seconds", "identity_check",
                      "measured_seconds", "duration_ratio", "review_reasons", "passes_initial_checks"]:
            assert parsed[field] == saved[field], f"{pid}: {field}"
        sample_stats[pid] = stats
        if "," in stats["NoS"]:
            noise_rows.append({"page_id": pid, "original": stats["NoS"], "converted": parsed["shot_count"]})
    # Apply the same full-precision choice to every archived measurement.
    records.append(saved.copy())
    uid, decade = saved["tmdb_id"], saved["decade"]
    if not saved["passes_initial_checks"]:
        continue
    if uid not in chosen and sum(r["decade"] == decade for r in chosen.values()) >= target_per_decade:
        continue
    key = (abs(saved["duration_ratio"] - 1), pid)
    if uid not in chosen or key < (abs(chosen[uid]["duration_ratio"] - 1), chosen[uid]["page_id"]):
        chosen[uid] = saved.copy()
assert sample_ids <= set(sample_stats)
print(f"{len(records)} saved measurements; {len(sample_stats)} HTML pages parsed and matched to log hashes.")


505 saved measurements; 20 HTML pages parsed and matched to log hashes.


## Choose one page per film
Continue in recorded vote-ranked candidate order until each decade has 20 passing films. Among passing pages for the same film use full-precision `abs(duration_ratio - 1)`, then page ID ascending; retain alternatives and report failed pages.

In [5]:
sample = list(chosen.values())
counts = Counter(r["decade"] for r in sample)
assert len(records) == 505 and len({r["tmdb_id"] for r in records}) == 160
assert len(sample) == 100 and set(counts.values()) == {20}
pairs = sorted((r["tmdb_id"], r["page_id"]) for r in sample)
assert hashlib.sha256(json.dumps(pairs, separators=(",", ":")).encode()).hexdigest() == film_version["frozen_pairs_sha256"]
for r in records:
    r["selected_for_main_analysis"] = (r["tmdb_id"], r["page_id"]) in pairs
# No discovered page before a selected rank boundary may be skipped.
parsed = {r["page_id"] for r in records}
boundary = {d: max((films[r["tmdb_id"]]["page"], films[r["tmdb_id"]]["rank_on_page"]) for r in sample if r["decade"] == d) for d in counts}
for p in inputs["pages"]:
    f = films[p["tmdb_id"]]
    if (f["page"], f["rank_on_page"]) <= boundary[f["decade"]]:
        assert p["page_id"] in parsed, "Higher-ranked page not checked"
display(pd.Series(counts, name="Frozen films").sort_index().to_frame())


,Frozen films
1970s,20
1980s,20
1990s,20
2000s,20
2010s,20


## Check shot-count formatting
A thousands comma is a display convention, not proof of a wrong source number. Reuse the parsed NoS values to count affected measurement pages; the frozen selected-page denominator is separate.

In [6]:
# Full counts were measured before the archive was reduced.
display(pd.DataFrame([
    ["Original measurement pages", compact["original_comma_pages"], len(records)],
    ["Original selected pages", compact["selected_comma_pages"], len(sample)],
    ["Reparsed pages", len(noise_rows), len(sample_stats)]
], columns=["Scope", "Pages with a comma", "Pages checked"]))
display(pd.DataFrame(noise_rows).head(1))
# Check saved title text before case-folded identity matching.
titles = pd.DataFrame(records)[["title_html", "title"]]
case_only = titles.title_html.ne(titles.title) & titles.title_html.str.casefold().eq(titles.title.str.casefold())
print(f"Title differs only by case on {case_only.sum()}/{len(titles)} pages.")
print("Example (HTML / TMDB):", *titles.loc[case_only].iloc[0], sep=" | ")
for field in titles:
    text = titles[field]
    print(field, "outer spaces:", text.ne(text.str.strip()).sum(),
          "nonprintable:", (~text.map(str.isprintable)).sum(), "replacement characters:", text.str.contains("�").sum())
print("Pages with invalid core numbers:", sum(not row["numeric_check"] for row in records))

,Scope,Pages with a comma,Pages checked
0,Original measurement pages,209,505
1,Original selected pages,82,100
2,Reparsed pages,8,20


,page_id,original,converted
0,500b5dac-563f-43f1-a85e-96fda98eaed4,"1,897",1897


Title differs only by case on 63/505 pages.
Example (HTML / TMDB): | The godfather | The Godfather
title_html outer spaces: 0 nonprintable: 0 replacement characters: 0
title outer spaces: 0 nonprintable: 0 replacement characters: 0
Pages with invalid core numbers: 0


## Check repeated measurements
Reuse the replayed measurements and frozen film list. Select the first eligible film by TMDB ID, then check the original full-precision page-choice rule; this is the American Beauty example.
For each parsed page, compare the original response hashes recorded in the request log, its entire 48-field merged row (including IDs), and its page ID. Count extra occurrences after the first. Different IDs can describe the same film.

In [7]:
rows = frozen_source["records"]
assert Counter(r["page_id"] for r in records) == Counter(r["page_id"] for r in rows)
html_hashes = []
row_keys = []
for row in rows:
    html_hashes.append(good_logs[row["url"]]["sha256"])
    row_keys.append(json.dumps(row, sort_keys=True, separators=(",", ":")))
page_ids = [row["page_id"] for row in rows]
duplicate_rows = []
for label, keys in [("Logged HTML SHA-256", html_hashes), ("Full merged row", row_keys), ("Page ID", page_ids)]:
    extra = len(keys) - len(set(keys))
    duplicate_rows.append([label, len(keys), extra, extra / len(keys)])
display(pd.DataFrame(duplicate_rows, columns=["Match on", "Pages", "Extra copies", "Rate"]))

,Match on,Pages,Extra copies,Rate
0,Logged HTML SHA-256,505,0,0.0
1,Full merged row,505,0,0.0
2,Page ID,505,0,0.0


In [8]:
case_id = next(uid for uid in sorted(chosen) if sum(r["tmdb_id"] == uid and r["passes_initial_checks"] for r in records) >= 2)
ranked = sorted([r for r in records if r["tmdb_id"] == case_id and r["passes_initial_checks"]],
                key=lambda r: (abs(r["duration_ratio"] - 1), r["page_id"]))
assert ranked[0]["page_id"] == chosen[case_id]["page_id"]
case_table = pd.DataFrame(ranked)[["title", "page_id", "duration_ratio", "asl_seconds", "selected_for_main_analysis"]]
case_table["Distance from one"] = [abs(r["duration_ratio"] - 1) for r in ranked]
with pd.option_context("display.precision", 17):
    display(case_table)
print("Pages at minimum distance:", sum(abs(r["duration_ratio"] - 1) == abs(ranked[0]["duration_ratio"] - 1) for r in ranked))


,title,page_id,duration_ratio,asl_seconds,selected_for_main_analysis,Distance from one
0,American Beauty,e41d20f3-5d12-44ed-a3a6-9be3cfa7f203,0.94703551912568307,7.00000000000000000,True,0.05296448087431693
1,American Beauty,ff48d25a-60c8-47ae-a494-448729d38a38,0.94635245901639342,6.70000000000000018,False,0.05364754098360658


Pages at minimum distance: 1


## Verify logs and the passport
Count all original attempts and successes. The HTML byte total comes from the original archive inventory.

In [9]:
passport = json.loads((folder / "data_passport.json").read_text(encoding="utf-8"))
log_summary = []
for channel, info in passport["main_runs"].items():
    path = folder / info["run"]
    logs = [json.loads(line) for line in (path / "requests.jsonl").read_text(encoding="utf-8").splitlines()]
    good = sum(200 <= r.get("status", 0) < 300 for r in logs)
    received_bytes = compact["original_html_bytes"] if channel == "HTML" else sum((path / r["file"]).stat().st_size for r in logs if r.get("file"))
    assert (len(logs), good, received_bytes) == (info["attempts"], info["successful_responses"], info["received_response_bytes"])
    log_summary.append([channel, len(logs), good, good / len(logs)])
assert passport["measurement_records"] == len(records)
display(pd.DataFrame(log_summary, columns=["Channel", "Attempts", "Successes", "Success rate"]))


,Channel,Attempts,Successes,Success rate
0,REST_API,383,373,0.973890
1,HTML,513,505,0.984405


## Check arithmetic against HTML
For selected pages in the raw sample, compare duration divided by shot count with ASL at the precision printed by the site. The version record also retains the original 100-film check.

In [10]:
audit_rows = []
for r in sample:
    if r["page_id"] not in sample_stats:
        continue
    stats = sample_stats[r["page_id"]]
    # Allow half a displayed unit for ASL and measured time.
    values = [int(stats["NoS"].replace(",", "")), seconds(stats["LEN"]), float(stats["ASL"]), float(stats["MSL"])]
    fields = ["shot_count", "measured_seconds", "asl_seconds", "msl_seconds"]
    assert all(math.isclose(r[k], v, rel_tol=0, abs_tol=1e-8) for k, v in zip(fields, values))
    asl_unit = float(Decimal(10) ** Decimal(stats["ASL"]).as_tuple().exponent)
    time_unit = float(Decimal(10) ** Decimal(stats["LEN"].split(":")[-1]).as_tuple().exponent)
    tolerance = asl_unit / 2 + time_unit / (2 * r["shot_count"])
    assert abs(r["measured_seconds"] / r["shot_count"] - r["asl_seconds"]) <= tolerance + 1e-10
    audit_rows.append([r["tmdb_id"], r["page_id"], "pass"])
measurement_audit = pd.DataFrame(audit_rows, columns=["tmdb_id", "page_id", "Arithmetic check"])
print("Selected raw sample pages passing arithmetic:", len(measurement_audit))

print("Original selected pages passing arithmetic (version record):", film_version["arithmetic_passed"])

Selected raw sample pages passing arithmetic: 8
Original selected pages passing arithmetic (version record): 100


## Verify the frozen table
Compare reconstructed values and page choices with the frozen CSV, and check all original exclusion reasons.

In [11]:
film_frame = pd.DataFrame(sample).sort_values(["decade", "tmdb_id"]).reset_index(drop=True)
film_frame["release_year"] = pd.to_datetime(film_frame["release_date"]).dt.year
assert film_frame["release_year"].eq(film_frame["year_html"]).all()
film_table = film_frame[frozen_main.columns]
pd.testing.assert_frame_equal(film_table, frozen_main, check_dtype=False)
prior = {r["page_id"]: r for r in frozen_source["records"]}
assert all(r["review_reasons"] == prior[r["page_id"]]["review_reasons"] for r in records)
state = json.loads((run / "extension_state.json").read_text(encoding="utf-8"))
assert state["selected_pairs_sha256"] == film_version["frozen_pairs_sha256"]
assert len(films) == 260 and len(parsed) == 505
print("Frozen CSV and all 100 selected pairs match. Original exclusion reasons retained for 505 pages.")


Frozen CSV and all 100 selected pairs match. Original exclusion reasons retained for 505 pages.


## Continue in Analysis
Use Film_Analysis for Q1-Q3 and M3-M6. The sample is purposive and edition/playback speed remain unverified. QA permits representative raw evidence and considers smaller datasets conditionally on complete engineering and analysis; no grading outcome is assumed.

## Sources

HTML: [Cinemetrics](https://cinemetrics.uchicago.edu/). API metadata and TMDB credit are in `Film_Selection.ipynb`. Saved raw responses, selectors and request logs support replay and source checks.